<a href="https://colab.research.google.com/github/LiaIssakov/Portfolio_2025/blob/main/simple_linear_classifier_Fall25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hyperplanes, linear classifiers and linear separability

Our favourite quote again (as it was actually uttered)

> "*To deal with hyper-planes in a 14-dimensional space, visualize a 3-D space and say 'fourteen' to yourself very loudly. Everyone does it.*" – Geoffrey Hinton

# Goals:
- Get used to thinking about **hyperplanes** (in spaces of arbitrary dimension)
- Get familiar with *using* simple **linear classifiers** (based on hyperplanes)
- Get used to thinking about **supervised learning** methods (the labels will play a crucial role now!)
- Use a linear classifier to check if images of 2 different mnist digits are **linearly separable** (namely, if they can be separated by a hyperplane)

# Non goals:
- Understand linear classifiers in depth and algorithms for computing them. The first step would be to understand the hinge loss, which can be minimized using gradient descent. This analysis would be similar to how we analyze the tSNE loss. We're not doing this though.

> But one thing to know is that unlike k-means and tSNE, standard linear classifiers do not get stuck in local minima! And as you'll see they are really fast to compute.

> Also often a nonlinear **kernel** is used to embed data in a higher-dimensional space where a linear (affine) decision boundary is used. This corresponds to a nonlinear decision boundary in the original space. We won't talk about this.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.svm import SVC, LinearSVC # new!
from keras.datasets import mnist as data

# Play with the visualization, try to get decent accuracy

There are two labels/classes: red, green. Misclasified points are drawn in lighter color.

> It may be easier to start from rotation.

> BTW. The loss used for this is not the accuracy! See the last extra task if curious.

In [ ]:
# this is used to predict labels in the visualization
def classify(P, v, intercept):
  dot_products = np.dot(P, v) + intercept
  return dot_products > 0 # True for positive, False for negative

In [ ]:
# just technicalities -- play with the interactive plot
from numpy.linalg import norm

from ipywidgets import *
import ipywidgets as widgets

from matplotlib.lines import Line2D

def vec2d(x,y): return np.array((x,y)) # returns a 2D vector as an np.array
def normalized(v): return v / norm(v) # normalizes a vector (makes it a unit vector)

def draw_point(v, txt = "", color = 'b'):
  '''I just plot a point. No need to look inside.'''
  plt.scatter(*v, c = color)
  plt.axis('equal') # otherwise it looks ugly
  if txt:
    plt.text(*(v + np.array((0.01, 0))), txt, c = 'gray', fontsize = 20)


def draw_arrow(v, t, txt = "", color = 'g', bold = False):
  '''I plot a nice arrow between v,t. No need to look inside, just try me out.'''
  v = np.asarray(v)
  t = np.asarray(t)
  dxy = t-v
  plt.scatter(*v, c = 'k')
  plt.scatter(*(v+dxy), c = 'k', alpha = 0)
  head_width = np.linalg.norm(dxy)/20. * [1,2.5][bold]
  line_width = 1 if not bold else 2
  plt.arrow(*v, *dxy, length_includes_head = True,
            head_width = head_width, linewidth = line_width, color = color)
  if txt:
    plt.text(*(v+dxy/2. + (0.01, 0)), txt, c = 'gray', fontsize = 20)
  plt.axis('equal') # otherwise it looks ugly

ORIGIN = vec2d(0,0) # we will use it to refer to the origin of R^2

sl_interc = lambda x : widgets.FloatSlider(value=x,
                                        min = -3, max = 3, step = 0.01)
sl_ang = lambda x : widgets.FloatSlider(value=x,
                                        min = -2*np.pi, max = 2*np.pi, step = 0.01)

np.random.seed(0)
A = np.random.randn(1000, 2)
B = np.random.randn(1000, 2) + (3.2,-2.5)

X = np.concatenate((A,B))
n = len(X)
labels = np.zeros(n)!=0
labels[np.arange(n) < n//2] = True

def plot_dot(ang1, intercept):
  # plt.sca(ax)
  plt.cla()

  v = vec2d(np.cos(ang1), np.sin(ang1))

  displacement = v/norm(v)*intercept

  predictions = classify(X, v, intercept)

  good_positive_points = X[(predictions == labels) & labels]
  good_negative_points = X[(predictions == labels) & ~labels]

  plt.scatter(*X[labels].T, color='lightgreen', label='bad')
  plt.scatter(*X[~labels].T, color='salmon', label='bad')

  plt.scatter(*good_positive_points.T, color='green', label='good')
  plt.scatter(*good_negative_points.T, color='red', label='good')

  okay = (predictions==labels).mean()

  plt.title(f'accuracy = {okay}')
  plt.legend()

  new_center = ORIGIN - displacement
  new_tip = v - displacement

  draw_arrow(new_center, new_tip, color = 'black', bold = True)

  vp = vec2d(v[1], -v[0])

  xy = np.array([new_center + 100*vp, new_center-100*vp])

  decision_boundary = Line2D(*xy.T)
  plt.gca().add_line(decision_boundary)

  plt.show()


#fig = plt.figure(figsize = (8,8))
#ax = plt.gca()



In [ ]:
interactive_plot = interactive(plot_dot,
                               ang1 = sl_ang(0.5),
                               intercept = sl_interc(0.0),
                               )

interactive_plot

## Hyperplane

By hyperplane we mean a generalization of a 2-dimensional plane in 3-dimensional Euclidean space. (People often refer to this higher dimensional space as the 'ambient space'.) In $n$-dimensional space, a hyperplane has dimension $n-1$.

> And not 2! Also not $1$ and not $n$.

More formally, we can talk about hyperplane determined by a vector $v \in \mathbb{R}^n$, namely $$H_v = \{ x \in \mathbb{R}^n\ |\ \langle x,v \rangle = 0\}.$$

This hyperplane splits $\mathbb{R}^n$ into the positive and negative halfspaces:
$$ H_v^+ = \{ x \in \mathbb{R}^n\ |\ \langle x,v \rangle > 0\},$$ and $$ H_v^- = \{ x \in \mathbb{R}^n\ |\ \langle x,v \rangle < 0\}.$$

> So this is what we did in 2D last time! The green and red points!


## Linear classifiers based on hyperplanes

We are in supervised learning setting. So we will use the correct labels (y_train) for training.

The goal is to learn to predict labels on the training images. We will then compare our prediction on the test images (x_train) with the y_test labels.

We will be using a **binary linear clasisfier**, which ideally tries to seperate the space into two parts: with 'positive' labels and 'negative' labels.

> One way of thinking about it, is to imagine a vector (like we did last time with red and green points, but in high dimension). We can think about the dot product of each input point and this vector. **The classifier tries to find the vector which gives the best separation between the two classes.**

In other words, it computes $v$ s.t. points in one class is (mostly) contained in $H_v^+$ and points in the other class at mostly contained in $H_v^+$.

In this case the hyperplane $H_v$ is often called the **decision boundary**.

> If the points in two classes can be perfectly separated by a hyperplane, they are called **linearly separable**. It usually means that the dataset is very easy to deal with!

> Most linear classifiers also train the **intercept** (aka **bias**) which effectively shifts the hyperplane, so that it does not have to pass the origin.

> Mnist has 10 classes, so we'll pick any two and see if they are linearly separable (or to what extent, they may be some outliers).

# Question:

How many parameters do we need to tune during training of such a classifier for data in $\mathbb{R}^d$? Suppose we do not use the intersept.

# Let's look at some (more) real data

## This just prepares the data (we need to extract data for two digits)

In [ ]:
(x_train, y_train), (x_test, y_test) = data.load_data()

x_train = x_train / 255.

# this looks complicated but I'm just selecting images of two digits,
# and the corresponding labels
D1, D2 = 6, 7

okay_train = (y_train==D1) | (y_train==D2) # 'or' of two masks
okay_test = (y_test==D1) | (y_test==D2)

x_train = x_train[okay_train]
y_train = y_train[okay_train]
x_test =  x_test[okay_test]
y_test = y_test[okay_test]

In [ ]:
plt.imshow(np.hstack(x_train[:8])) # just a trick to plot some images in a row

In [ ]:
'''With these parameters linear_classifier_1 is very close to the
vanilla linear classifier we discuss in class with the vector
passing through the origin
'''
linear_classifier_1 = LinearSVC(loss='hinge', penalty='l2',
                              fit_intercept=False,
                              max_iter=5000, C=1e30)


'''This one is similar, but uses the intercept. But it gives access to the
support vectors, which is nice.
Let's use this one.
'''
linear_classifier_2 = SVC(kernel='linear')

# Experiment: are two classes of mnist linearly separable?


## Task 1: Apply linear classifier and check the accuracy

Fit the classifier with data, predict the labels on the test examples, and compare with the test labels. What's the accuracy?

> Hint: remember that **all model from sklearn follow the same interface** (like tSNE, kMeans...) But now we will also use the correct labels!

> Hint: accuracy is just 1 - #errors / #examples = #good / #examples.

> For now we can implement it using np.mean. We'll use proper functions later.

You can also check how it performed on the training examples, but that's generally less interesting. We want our model to generalize.

In [ ]:
x_train.shape, y_train.shape

In [ ]:
# fit


In [ ]:
# predict (use the .predict function twice (!) on your classifier, similarly to .fit)
# store preds and train_preds variables, for what's next


In [ ]:
y_test.shape

In [ ]:
# get accuracy, just check which are good, compute np.mean
# or just use .score
def compute_accuracy(predictions, correct):
  # counts the correct predictions, divides by n
  return np.mean(predictions==correct)

print('test acc', compute_accuracy(preds, y_test))
print('train acc', compute_accuracy(train_preds, y_train))

## Task 2:

Plot the misclassified test images and their (mis)predictions and labels.

In [ ]:
# preds = 1-y_test

# Visualizing the support vectors

One classical way of computing linear classifiers is using so-called Support Vector Machines (SVMs). We actually use them: SVC stands for Support Vector Classifier.

The support vectors are the vectors which actually affect (support) the (margins) of the computed decision hyperplane.

> It makes sense that vectors which are far away from the decision hyperplane are kind of irrelevant.

Let's try to visualize these support vectors, since SVC gives us access to them.

> What would you expect to see?

In [ ]:
# how many of them?
indices_of_support_vectors.shape

In [ ]:
indices_of_support_vectors = linear_classifier_2.support_

support_vectors = x_train[indices_of_support_vectors]

# let's plot some of them
for im in support_vectors[::20]:
  plt.imshow(im)
  plt.show()

# Extra Task:

The code below accesses the vector which is used to classify the examples.

Try to use this vector $v$ to classify each example 'by hand' (with your own implementation) in the test dataset. You should get the same results as in task 1 -- so this confirms what the classifier does internally.

> Tip: this may seem complicated, but it's very similar as the task with red and green points last time!

> Top: for linear_classifier_2 you'll need to add the intercept_ as well. If this is confusing, use linear_classifier_1.

> Tip: You may have to check which class corresponds to the 'positive' example and which to the 'negative' ones.

In [ ]:
v = linear_classifier_1.coef_.flatten()
# ...

# Extra bonus task:

This classifier minimizes the so-called **hinge loss**. Try to do a little research on it -- it's really neat. Maybe you even have time to implement your version of the classifier by minimizing the hinge loss?

> Tip: you can use from function minimize from scipy.optimize I briefly showed you some classes ago.